# MID training — single scene

End-to-end training of the MID (Motion Indeterminacy Diffusion) trajectory model on **one** ETH/UCY scene. Once this works for one scene, the same notebook generalizes to leave-one-out cross-validation by loading the merged train environment.

**Pipeline:**
1. Load preprocessed `Environment` from `processed_data/<scene>_train.pkl`
2. Construct the Trajectron++ encoder (imported from `MID/`)
3. Wrap it with our `AutoEncoder` (encoder + DDPM noise predictor)
4. Train with the standard DDPM ε-prediction objective
5. Save a checkpoint

Evaluation (ADE/FDE Best-of-20) is in a separate notebook.

## 1. Configuration

In [ ]:
SCENE = "eth"            # one of: eth, hotel, univ, zara1, zara2
BATCH_SIZE = 256
EPOCHS = 5               # smoke-test value; paper uses 90
LR = 1e-3
ENCODER_DIM = 256
TF_LAYER = 3
NUM_DIFFUSION_STEPS = 100
AUGMENT = False          # set True once you've verified training works end-to-end
SEED = 123

CHECKPOINT_DIR = "../checkpoints"
CHECKPOINT_NAME = f"mid_{SCENE}.pt"

## 2. Setup: paths, device, seeds

`PROJECT_ROOT` (this notebook's parent directory) needs to be on `sys.path` so `import mid_model`, `import environment`, `import models`, `import dataset`, and `import utils` all resolve to the local copies at the project root. The MID repo at `MID/` is no longer needed at runtime — everything we use has been pulled out.

In [ ]:
import os
import sys
import time
import random
import numpy as np
import torch

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Pick the best device available.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 3. Load preprocessed Environment + build DataLoader

`load_environment` un-dills the file produced by `pre-process.ipynb`. `build_dataloader` walks the Environment's scenes, indexes every valid (scene, t, node) tuple, and returns a `DataLoader` that yields batches the encoder can consume directly.

The batch is a 9-tuple: `(first_history_index, x_t, y_t, x_st_t, y_st_t, neighbors_data_st, neighbors_edge_value, robot_traj_st_t, map)`. We pass the whole tuple to `encoder.get_latent(...)`; only `y_t` is consumed by the diffusion model (as the clean target trajectory).

In [ ]:
from mid_model import load_environment, build_dataloader, get_hyperparameters

pkl_path = os.path.join(PROJECT_ROOT, "processed_data", f"{SCENE}_train.pkl")
env = load_environment(pkl_path)
print(f"Loaded {pkl_path}")
print(f"  scenes: {len(env.scenes)}")
print(f"  total nodes: {sum(len(s.nodes) for s in env.scenes)}")
print(f"  attention_radius: {env.attention_radius}")

hyperparams = get_hyperparameters(encoder_dim=ENCODER_DIM)
hyperparams["batch_size"] = BATCH_SIZE

train_loader, node_type = build_dataloader(
    env=env,
    hyperparams=hyperparams,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    augment=AUGMENT,
)
print(f"\nNode type: {node_type}")
print(f"Batches per epoch: {len(train_loader)}")

## 4. Construct the Trajectron++ encoder (imported)

We don't reimplement the encoder — it's a deeply nested CVAE with social-attention edges that already works. We import it from `MID/models/trajectron.py` and configure it with the standard Trajectron++ sequence:

```
registrar = ModelRegistrar(model_dir, device)
encoder   = Trajectron(registrar, hyperparams, device)
encoder.set_environment(env)         # builds one MGCVAE per node type
encoder.set_annealing_params()       # KL/tau schedulers (inert in our path)
```

`ModelRegistrar` is an `nn.Module` that owns the encoder's sub-models. Our `AutoEncoder` re-registers it so `model.parameters()` covers both the encoder and the diffusion net.

In [ ]:
from utils.model_registrar import ModelRegistrar
from models.trajectron import Trajectron

registrar = ModelRegistrar(model_dir=CHECKPOINT_DIR, device=DEVICE)
encoder = Trajectron(registrar, hyperparams, DEVICE)
encoder.set_environment(env)
encoder.set_annealing_params()

print(f"Encoder constructed with {sum(p.numel() for p in registrar.parameters()):,} parameters")

## 5. Assemble the full model

In [ ]:
from mid_model import AutoEncoder

model = AutoEncoder(
    encoder=encoder,
    registrar=registrar,
    encoder_dim=ENCODER_DIM,
    num_diffusion_steps=NUM_DIFFUSION_STEPS,
    beta_1=1e-4,
    beta_T=5e-2,
    tf_layer=TF_LAYER,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable:,}")

## 6. Smoke test: one batch, one forward pass

Before committing to a full training run, do one forward pass and check that the loss is a finite scalar. This is where shape mismatches and device errors surface.

In [ ]:
batch = next(iter(train_loader))
model.train()
loss = model.get_loss(batch, node_type)
print(f"Smoke-test loss: {loss.item():.4f}")
assert torch.isfinite(loss), "loss is not finite — something is off"

## 7. Training loop

Standard supervised loop. One optimizer over `model.parameters()` covers both the encoder and the diffusion network — they're trained jointly, as in the paper.

We keep this minimal: Adam, no LR schedule for now, no gradient clipping. The paper's reported numbers come from 90 epochs; for a smoke test 5–10 is enough to see the loss come down.

In [ ]:
from tqdm.auto import tqdm

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = []
for epoch in range(1, EPOCHS + 1):
    epoch_losses = []
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", ncols=100)
    for batch in pbar:
        optimizer.zero_grad()
        loss = model.get_loss(batch, node_type)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = float(np.mean(epoch_losses))
    elapsed = time.time() - t0
    history.append(avg_loss)
    print(f"  epoch {epoch}: avg loss = {avg_loss:.4f}  ({elapsed:.1f}s)")

## 8. Save checkpoint

Save the encoder's `ModelRegistrar` and the diffusion network separately. The encoder is heavyweight (LSTMs, edge models) and useful to load standalone for analysis. The diffusion net is small.

In [ ]:
ckpt_path = os.path.join(CHECKPOINT_DIR, CHECKPOINT_NAME)
torch.save({
    "scene": SCENE,
    "epoch": EPOCHS,
    "hyperparams": hyperparams,
    "encoder_dim": ENCODER_DIM,
    "tf_layer": TF_LAYER,
    "num_diffusion_steps": NUM_DIFFUSION_STEPS,
    "registrar_state_dict": registrar.model_dict.state_dict(),
    "diffusion_state_dict": model.diffusion.state_dict(),
    "history": history,
}, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")

## Next steps

- Verify training loss is decreasing across epochs (sanity check the implementation)
- Enable `AUGMENT=True` for the full training run (uses the 24 pre-computed rotations from `pre-process.ipynb`)
- Bump `EPOCHS` to 90 once the smoke test looks good
- Build the evaluation notebook (Best-of-20 ADE/FDE)
- Train on the other 4 scenes / merge for leave-one-out